# jaxfne — Étude No. 10 · Global/Local Oddball Task

Classic global-local design (Bekinschtein-style): 5-tone sequences ending
either in a repeat of the first tone (`AAAAA`, "local standard") or a
different tone (`AAAAB`, "local deviant"). Each of two blocks makes ONE of
these sequence types frequent (block expectation) and the other rare
(block violation) — giving 4 cells: local x global, each independently
varying whether the LAST tone matches the preceding 4 (local) and whether
the sequence type matches the block's dominant pattern (global).

## 1. Setup

In [1]:
import numpy as np
import jax.numpy as jnp
import jaxfne as jtfne

print("jaxfne", jtfne.__version__)


jaxfne 0.4.4


## 2. Config

In [2]:
N_NEURONS = 200
SEED = 0
DT_MS = 0.5
FX_MS = 200.0         # fixation before the first tone
P_DUR_MS = 100.0       # tone duration
ISI_MS = 300.0         # inter-tone interval
N_TONES = 5            # tones per sequence
TOTAL_MS = FX_MS + N_TONES * ISI_MS
N_FREQUENT_TRIALS = 15  # trials of the block's dominant sequence type
N_RARE_TRIALS = 5       # trials of the block's rare sequence type

cfg = (
    jtfne.build_laminar_column(name="V1", n=N_NEURONS, ei_profile="canonical")
    .runtime(seed=SEED, recurrent_backend="edge_list")
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "V_m"], n_contacts=8)
    .field(domain="laminar_column", conductivity="proxy", boundary="mean_zero_neumann")
)
model = jtfne.construct(cfg)
nt = model.neuron_table()
l4e_idx = [i for i, r in enumerate(nt) if r.get("layer") == "L4" and r.get("cell_type") == "E"]
print(f"L4 E neurons: {len(l4e_idx)} / {N_NEURONS}")


L4 E neurons: 14 / 200


## 3. Stimulus schedules

Two sequence templates: `AAAAA` (local standard, all 5 tones identical
amplitude) and `AAAAB` (local deviant, 5th tone raised amplitude). Each
trial is wrapped in `jaxfne.StimulusSchedule` — a bare list of event dicts
is silently ignored by `simulate()` (same gap documented in Étude 9;
`Model._resolve_stimulus_schedule` only recognizes
`StimulusSchedule`/`ParadigmCondition`).

Block assignment is deterministic (not randomly sampled) — 15 frequent +
5 rare trials per block, matching Étude 9's per-condition trial count —
so the rare cell is never accidentally empty.

In [3]:
def make_sequence_schedule(is_local_deviant):
    """AAAAA (False) or AAAAB (True): only the 5th tone's amplitude differs."""
    events = []
    for k in range(N_TONES):
        onset = FX_MS + k * ISI_MS
        is_final_deviant = is_local_deviant and (k == N_TONES - 1)
        events.append({
            "onset_ms": onset, "duration_ms": P_DUR_MS,
            "amplitude": 8.0 if is_final_deviant else 4.0,
            "label": "deviant" if is_final_deviant else "standard",
            "is_drive_event": True,
            "target_indices": l4e_idx,
        })
    return jtfne.StimulusSchedule(events=events, n_neurons=len(nt))

sched_local_standard = make_sequence_schedule(is_local_deviant=False)  # AAAAA
sched_local_deviant = make_sequence_schedule(is_local_deviant=True)    # AAAAB

# Block A: AAAAA frequent (global standard), AAAAB rare (global deviant)
# Block B: AAAAB frequent (global standard), AAAAA rare (global deviant)
block_trial_plan = {
    "block_A": (
        [("local_standard", sched_local_standard)] * N_FREQUENT_TRIALS
        + [("local_deviant", sched_local_deviant)] * N_RARE_TRIALS
    ),
    "block_B": (
        [("local_deviant", sched_local_deviant)] * N_FREQUENT_TRIALS
        + [("local_standard", sched_local_standard)] * N_RARE_TRIALS
    ),
}


## 4. Run

In [4]:
def post_final_tone_rate(sched, trial_seed):
    window_start = int((FX_MS + (N_TONES - 1) * ISI_MS) / DT_MS)
    window_end = int((FX_MS + (N_TONES - 1) * ISI_MS + P_DUR_MS + 100.0) / DT_MS)
    sig = jtfne.simulate(
        model, sim=jtfne.Simulation(duration_ms=TOTAL_MS, dt_ms=DT_MS, seed=trial_seed),
        paradigm=sched,
    )
    assert bool(jnp.all(jnp.isfinite(sig.V_m))), "non-finite V_m -- do not trust this run"
    spk = np.asarray(sig.spikes)
    return spk[window_start:window_end, l4e_idx].mean() * 1000.0 / DT_MS

results = {}  # (block, local_condition) -> np.array of per-trial rates
trial_seed = SEED
for block_name, plan in block_trial_plan.items():
    by_condition = {"local_standard": [], "local_deviant": []}
    for local_condition, sched in plan:
        by_condition[local_condition].append(post_final_tone_rate(sched, trial_seed))
        trial_seed += 1
    for local_condition, rates in by_condition.items():
        rates_arr = np.array(rates)
        results[(block_name, local_condition)] = rates_arr
        print(f"{block_name:8s} {local_condition:15s} n={len(rates_arr):2d}  "
              f"rate={rates_arr.mean():.2f}+-{rates_arr.std():.2f} Hz")


block_A  local_standard  n=15  rate=11.83+-0.69 Hz
block_A  local_deviant   n= 5  rate=13.00+-0.48 Hz


block_B  local_standard  n= 5  rate=12.00+-0.36 Hz
block_B  local_deviant   n=15  rate=13.43+-0.58 Hz


## 5. Objective — local and global mismatch responses

Local mismatch: deviant-final vs standard-final tone, pooled across both
blocks (matches Étude 9's contrast, generalized to sequence-level).

Global mismatch: rare-in-block vs frequent-in-block, computed separately
for each local condition (block_B's `local_deviant` trials are frequent
there / block_A's are rare there, and vice versa for `local_standard`) —
isolates the block-frequency violation effect while holding the local
(within-sequence) structure fixed.

In [5]:
local_deviant_rates = np.concatenate([results[("block_A", "local_deviant")], results[("block_B", "local_deviant")]])
local_standard_rates = np.concatenate([results[("block_A", "local_standard")], results[("block_B", "local_standard")]])
local_mismatch_hz = float(local_deviant_rates.mean() - local_standard_rates.mean())

# global mismatch for local_deviant sequences: rare (block_A) - frequent (block_B)
global_mismatch_local_deviant_hz = float(
    results[("block_A", "local_deviant")].mean() - results[("block_B", "local_deviant")].mean()
)
# global mismatch for local_standard sequences: rare (block_B) - frequent (block_A)
global_mismatch_local_standard_hz = float(
    results[("block_B", "local_standard")].mean() - results[("block_A", "local_standard")].mean()
)

print(f"local mismatch (deviant - standard, pooled): {local_mismatch_hz:+.2f} Hz")
print(f"global mismatch (rare - frequent | local_deviant): {global_mismatch_local_deviant_hz:+.2f} Hz")
print(f"global mismatch (rare - frequent | local_standard): {global_mismatch_local_standard_hz:+.2f} Hz")

# Computational diagnostic, not a claim of biological global/local MMN --
# a same-model firing-rate comparison across sequence templates and block
# assignments, not a validated ERP/MMN result.
assert np.isfinite(local_mismatch_hz)
assert np.isfinite(global_mismatch_local_deviant_hz)
assert np.isfinite(global_mismatch_local_standard_hz)


local mismatch (deviant - standard, pooled): +1.45 Hz
global mismatch (rare - frequent | local_deviant): -0.43 Hz
global mismatch (rare - frequent | local_standard): +0.17 Hz


## 6. Export

In [6]:
import json as _json
from pathlib import Path

OUT_DIR = Path("local/etude10")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "notebook": "jaxfne_etude_no_10_global_local_oddball",
    "jaxfne_version": jtfne.__version__,
    "paradigm": {
        "name": "global_local_oddball", "n_tones": N_TONES,
        "p_dur_ms": P_DUR_MS, "isi_ms": ISI_MS, "fixation_ms": FX_MS,
        "n_frequent_trials": N_FREQUENT_TRIALS, "n_rare_trials": N_RARE_TRIALS,
        "stimulus_target": "L4_E_neurons", "n_stimulus_targets": len(l4e_idx),
    },
    "config": {"N_NEURONS": N_NEURONS, "DT_MS": DT_MS, "SEED": SEED},
    "results": {
        "local_mismatch_hz": local_mismatch_hz,
        "global_mismatch_local_deviant_hz": global_mismatch_local_deviant_hz,
        "global_mismatch_local_standard_hz": global_mismatch_local_standard_hz,
        "cell_rates_hz_mean": {f"{b}/{c}": float(v.mean()) for (b, c), v in results.items()},
        "cell_rates_hz_std": {f"{b}/{c}": float(v.std()) for (b, c), v in results.items()},
    },
}
(OUT_DIR / "manifest.json").write_text(_json.dumps(manifest, indent=2))
print("wrote", OUT_DIR / "manifest.json")


wrote local/etude10/manifest.json
